# California General Land Use Data Gather

In [0]:
import requests

from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.getOrCreate()

query_url = "https://services8.arcgis.com/Xr1lDrwMv89PhjD9/arcgis/rest/services/California_General_Plan_Land_Use/FeatureServer/2/query?where=1%3D1&outFields=*&outSR=4326&f=json"

ca_land_response = requests.get(query_url)
print(f"Status code: {ca_land_response.status_code}")

ca_land_data = ca_land_response.json()




In [0]:
ca_land_pandas_df = pd.json_normalize(ca_land_data['features'])
ca_land_pandas_df.head()

In [0]:
ca_land_pandas_df.columns

In [0]:
col_rename_dict = {
    'attributes.OBJECTID': 'OBJECTID',
    'attributes.County': 'County',
    'attributes.jurisdiction': 'jurisdiction',
    'attributes.classkey': 'classkey',
    'attributes.code': 'code',
    'attributes.description': 'description',
    'attributes.ucd_number': 'ucd_number',
    'attributes.ucd_description': 'ucd_description',
    'attributes.Source': 'Source',
    'attributes.Date': 'Date',
    'attributes.Shape__Area': 'Shape__Area',
    'attributes.Shape__Length': 'Shape__Length',
    'geometry.rings': 'rings'
}


In [0]:
ca_land_pandas_df.rename(columns=col_rename_dict, inplace=True)
ca_land_pandas_df.head()


In [0]:
ca_land_spark_df = spark.createDataFrame(ca_land_pandas_df)
ca_land_spark_df.printSchema()


In [0]:
delete_table = "DROP TABLE IF EXISTS ca_healthcare_fac_bronze.ca_general_land_use_data.ca_land_use_df"
spark.sql(delete_table)

In [0]:
ca_land_spark_df.write.mode("append").saveAsTable("ca_healthcare_fac_bronze.ca_general_land_use_data.ca_land_use_df")